# DeepSeek 训练计划生成测试

把 `DEEPSEEK_API_KEY` 填到 `backend/.env` 后运行。这个 Notebook 用于独立验证 DeepSeek 是否能按 AI-FIT 的 JSON schema 返回训练计划。

In [ ]:
import json
import os
import urllib.request
from pathlib import Path

env_path = Path(r"E:\AI-Fit\AI-Fit\backend\.env")
for line in env_path.read_text(encoding="utf-8-sig").splitlines():
    if "=" in line and not line.strip().startswith("#"):
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())

api_key = os.environ.get("DEEPSEEK_API_KEY", "")
base_url = os.environ.get("DEEPSEEK_BASE_URL", "https://api.deepseek.com").rstrip("/")
model = os.environ.get("DEEPSEEK_MODEL", "deepseek-v4-flash")
print({"configured": bool(api_key), "base_url": base_url, "model": model})

In [ ]:
if not api_key:
    raise RuntimeError("请先在 backend/.env 填写 DEEPSEEK_API_KEY")

supported_equipment = [
    "Chest Press machine", "Lat Pull Down", "Seated Cable Rows", "arm curl machine",
    "chest fly machine", "chinning dipping", "lateral raises machine", "leg extension",
    "leg press", "reg curl machine", "seated dip machine", "shoulder press machine", "smith machine",
]
payload = {
    "model": model,
    "messages": [
        {"role": "system", "content": "你是 AI-FIT 健身计划生成引擎。只输出 JSON。动作必须来自 supportedEquipment。"},
        {"role": "user", "content": json.dumps({
            "goal": "增肌",
            "trainingPlace": "健身房固定器械区",
            "weeklyFrequency": "每周 4 次",
            "sessionDuration": "45 分钟",
            "focusPreference": "背部发力",
            "supportedEquipment": supported_equipment,
        }, ensure_ascii=False)},
    ],
    "thinking": {"type": "disabled"},
    "response_format": {"type": "json_object"},
}

request = urllib.request.Request(
    f"{base_url}/chat/completions",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
    method="POST",
)
with urllib.request.urlopen(request, timeout=45) as response:
    data = json.loads(response.read().decode("utf-8"))

content = data["choices"][0]["message"]["content"]
print(content)